<a href="https://colab.research.google.com/github/MalekiMostafa/DeepLearning1403/blob/Project/Project3_TextClassifierWithAttention/DL_Project3_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Report:
# Text Classification Using Attention Mechanism

## Objective
The objective of this project is to build a text classification model using the Attention Mechanism. The developed model should take an input text and classify it into one of the predefined categories.

## Project Steps

### 1. Data Loading and Preprocessing
- **Dataset**: The Consumer Complaints dataset was used, which contains consumer complaints regarding various financial products.
- **Preprocessing Steps**:
  - Removing rows without complaint text.
  - Converting text to lowercase.
  - Removing punctuation, numbers, and unnecessary characters.
  - Tokenizing text and converting it into numerical sequences using a vocabulary created from GloVe embeddings.
  - Padding sequences to a fixed length (20 tokens).

### 2. Loading GloVe Embeddings
- Pre-trained **GloVe embeddings** with 300 dimensions were used.
- The `glove.42B.300d.zip` file was downloaded and extracted.
- Embeddings were stored along with the corresponding vocabulary.

### 3. Encoding Labels
- Class labels (e.g., **Credit card, Mortgage, Loan, etc.**) were converted into numerical values using **LabelEncoder**.

### 4. Building the Attention Model
- A neural network model with an **Attention Mechanism** was developed.
- The model includes the following layers:
  - **Attention Layer** to compute attention weights.
  - **Linear Layer** for final classification.
- **Activation Functions**: Tanh and Softmax.

### 5. Model Training
- The dataset was split into training, validation, and test sets.
- The model was trained for **50 epochs**.
- **Loss Function**: CrossEntropyLoss.
- **Optimizer**: Adam with a learning rate of **0.0005**.
- The model was evaluated on both **training and validation** data during each epoch, and the best model (based on lowest validation loss) was saved.

### 6. Model Evaluation
- The trained model was evaluated on the **test dataset**.
- **Test Accuracy**: **75.47%**.
- **Test Loss**: **0.748**.

### 7. Prediction on New Text
- A sample new text was given to the model, and it correctly classified it as **credit_report**.

## Results
- **Model Accuracy**: 75.47%
- **Test Loss**: 0.748
- **Training Time**: Approximately 1 hour and 40 minutes (depending on hardware).

## Challenges and Solutions
- **Challenge**: Large dataset and long training time.
  - **Solution**: Using GPU to speed up training.
- **Challenge**: Variability in text length.
  - **Solution**: Padding sequences to a fixed length (20 tokens).
- **Challenge**: Low initial accuracy.
  - **Solution**: Adjusting learning rate and increasing the number of epochs.

## Conclusion
This project demonstrated that using the Attention Mechanism can improve text classification accuracy. However, further improvements can be achieved by:
- Increasing the training dataset size.
- Using more advanced models like **Transformer**.
- Fine-tuning hyperparameters more precisely.

## Output Files
- `tokens.pkl`: Tokenized texts.
- `labels.pkl`: Encoded labels.
- `embeddings.pkl`: GloVe embeddings.
- `vocabulary.pkl`: Generated vocabulary.
- `label_encoder.pkl`: Label encoder.
- `attention.pth`: Trained model.

## Final Remarks
The developed model successfully classified input texts with **reasonable accuracy**. The use of the **Attention Mechanism** helped the model focus on important parts of the text, improving performance.



##Conclusion

This project demonstrates the effectiveness of using an attention mechanism for text classification tasks. By focusing on relevant parts of the input text, the attention model achieves improved accuracy in categorizing consumer complaints into predefined product categories.

In [1]:
import re
import torch
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [58]:
lr = 0.0005
vec_len = 300
seq_len = 20
num_epochs = 50
label_col = "Product"
tokens_path = "/content/drive/MyDrive/DL_Project3_Attention/tokens.pkl"
labels_path = "/content/drive/MyDrive/DL_Project3_Attention/labels.pkl"
data_path = "/content/drive/MyDrive/DL_Project3_Attention/Consumer_Complaints.csv"
model_path = "/content/drive/MyDrive/DL_Project3_Attention/attention.pth"
vocabulary_path = "/content/drive/MyDrive/DL_Project3_Attention/vocabulary.pkl"
embeddings_path = "/content/drive/MyDrive/DL_Project3_Attention/embeddings.pkl"
text_col_name = "Consumer Complaint"
label_encoder_path = "/content/drive/MyDrive/DL_Project3_Attention/label_encoder.pkl"
product_map = {'Vehicle loan or lease': 'vehicle_loan',
               'Credit reporting, credit repair services, or other personal consumer reports': 'credit_report',
               'Credit card or prepaid card': 'card',
               'Money transfer, virtual currency, or money service': 'money_transfer',
               'virtual currency': 'money_transfer',
               'Mortgage': 'mortgage',
               'Payday loan, title loan, or personal loan': 'loan',
               'Debt collection': 'debt_collection',
               'Checking or savings account': 'savings_account',
               'Credit card': 'card',
               'Bank account or service': 'savings_account',
               'Credit reporting': 'credit_report',
               'Prepaid card': 'card',
               'Payday loan': 'loan',
               'Other financial service': 'others',
               'Virtual currency': 'money_transfer',
               'Student loan': 'loan',
               'Consumer Loan': 'loan',
               'Money transfers': 'money_transfer'}

Process text data

In [4]:
# خواندن فایل CSV
data = pd.read_csv(data_path)

# نمایش ۵ ردیف اول داده‌ها
print(data.head())

  Date received           Product     Sub-product  \
0    03-12-2014          Mortgage  Other mortgage   
1    10-01-2016  Credit reporting             NaN   
2    10/17/2016     Consumer Loan    Vehicle loan   
3    06-08-2014       Credit card             NaN   
4    09/13/2014   Debt collection     Credit card   

                                      Issue                   Sub-issue  \
0  Loan modification,collection,foreclosure                         NaN   
1    Incorrect information on credit report              Account status   
2                Managing the loan or lease                         NaN   
3                                Bankruptcy                         NaN   
4                     Communication tactics  Frequent or repeated calls   

                                  Consumer Complaint  \
0                                                NaN   
1  I have outdated information on my credit repor...   
2  I purchased a new car on XXXX XXXX. The car de...   
3     

In [5]:
data.dropna(subset=[text_col_name], inplace=True)

data.replace({label_col: product_map}, inplace=True)

In [13]:
import os
import requests

# URL فایل GloVe
glove_url = "https://nlp.stanford.edu/data/glove.42B.300d.zip"

# مسیر ذخیره فایل دانلود شده
glove_zip_path = "/content/drive/MyDrive/DL_Project3_Attention/glove.42B.300d.zip"

# دانلود فایل
if not os.path.exists(glove_zip_path):
    print("در حال دانلود فایل GloVe...")
    response = requests.get(glove_url, stream=True)
    with open(glove_zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk:
                f.write(chunk)
    print("دانلود کامل شد!")
else:
    print("فایل از قبل دانلود شده است.")

# اکسترکت فایل ZIP
import zipfile

glove_extract_path = "/content/glove.42B.300d"
if not os.path.exists(glove_extract_path):
    print("در حال اکسترکت فایل...")
    with zipfile.ZipFile(glove_zip_path, 'r') as zip_ref:
        zip_ref.extractall(glove_extract_path)
    print("اکسترکت کامل شد!")
else:
    print("فایل از قبل اکسترکت شده است.")

# مسیر فایل glove.42B.300d.txt با استفاده از glove_vector_path
glove_vector_path = os.path.join(glove_extract_path, "glove.42B.300d.txt")

# بررسی وجود فایل
if os.path.exists(glove_vector_path):
    print(f"فایل GloVe در مسیر زیر ذخیره شده است: {glove_vector_path}")
else:
    print("فایل GloVe یافت نشد!")

فایل از قبل دانلود شده است.
در حال اکسترکت فایل...
اکسترکت کامل شد!
فایل GloVe در مسیر زیر ذخیره شده است: /content/glove.42B.300d/glove.42B.300d.txt


In [11]:
def save_file(name, obj):
    """
    Function to save an object as pickle file
    """
    with open(name, 'wb') as f:
        pickle.dump(obj, f)


def load_file(name):
    """
    Function to load a pickle object
    """
    return pickle.load(open(name, "rb"))

Process glove embeddings

In [14]:
with open(glove_vector_path, "rt", encoding="utf-8") as f:
    emb = f.readlines()


In [9]:
vocabulary, embeddings = [], []

for item in emb:
    vocabulary.append(item.split()[0])
    embeddings.append(item.split()[1:])

In [10]:
embeddings = np.array(embeddings, dtype=np.float32)

In [11]:
vocabulary = ["<pad>", "<unk>"] + vocabulary

In [12]:
import numpy as np

# تغییر شکل آرایه اول به (1, 300)
ones_array = np.ones((1, 300), dtype=np.float32)  # تغییر از ۵۰ به ۳۰۰

# اتصال آرایه‌ها
embeddings = np.vstack([ones_array,
                        np.mean(embeddings, axis=0).reshape(1, -1),  # میانگین بردارها
                        embeddings])

In [13]:
save_file(embeddings_path, embeddings)
save_file(vocabulary_path, vocabulary)

Encode labels

In [14]:
label_encoder = LabelEncoder()
label_encoder.fit(data[label_col])
labels = label_encoder.transform(data[label_col])

save_file(labels_path, labels)
save_file(label_encoder_path, label_encoder)


Process the text column

In [15]:
input_text = list(data[text_col_name])

len(input_text)

277814

Convert text to lower case

Remove punctuations except apostrophe

In [16]:
input_text = [i.lower() for i in tqdm(input_text)]

100%|██████████| 277814/277814 [00:00<00:00, 908720.13it/s]


In [17]:
input_text = [re.sub(r"[^\w\d'\s]+", " ", i)
              for i in tqdm(input_text)]

100%|██████████| 277814/277814 [00:08<00:00, 31600.36it/s]


Remove digits

In [18]:
input_text = [re.sub("\d+", "", i) for i in tqdm(input_text)]

100%|██████████| 277814/277814 [00:05<00:00, 50335.05it/s]


Remove more than one consecutive instance of 'x'

In [19]:
input_text = [re.sub(r'[x]{2,}', "", i) for i in tqdm(input_text)]

100%|██████████| 277814/277814 [00:04<00:00, 59411.32it/s]


Remove multiple spaces with single space

In [20]:
input_text = [re.sub(' +', ' ', i) for i in tqdm(input_text)]

100%|██████████| 277814/277814 [00:13<00:00, 21125.34it/s]


Tokenize the text

In [16]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [22]:
tokens = [word_tokenize(t) for t in tqdm(input_text)]

100%|██████████| 277814/277814 [02:03<00:00, 2257.05it/s]


Take the first 20 tokens in each complaint text

In [23]:
tokens = [i[:20] if len(i) > 19 else ['<pad>'] * (20 - len(i)) + i
          for i in tqdm(tokens)]

100%|██████████| 277814/277814 [00:03<00:00, 77994.45it/s]


Convert tokens to integer indices from vocabulary

In [24]:
def token_index(tokens, vocabulary, missing='<unk>'):
    """
    :param tokens: List of word tokens
    :param vocabulary: All words in the embeddings
    :param missing: Token for words not present in the vocabulary
    :return: List of integers representing the word tokens
    """
    idx_token = []
    for text in tqdm(tokens):
        idx_text = []
        for token in text:
            if token in vocabulary:
                idx_text.append(vocabulary.index(token))
            else:
                idx_text.append(vocabulary.index(missing))
        idx_token.append(idx_text)
    return idx_token

In [25]:
tokens = token_index(tokens, vocabulary)

100%|██████████| 277814/277814 [1:37:55<00:00, 47.28it/s]


Save the tokens

In [26]:
save_file(tokens_path, tokens)

Create attention model

In [60]:
class AttentionModel(nn.Module):

    def __init__(self, vec_len, seq_len, n_classes):
        super(AttentionModel, self).__init__()
        self.vec_len = vec_len
        self.seq_len = seq_len
        self.attn_weights = torch.cat([torch.tensor([[0.]]),
                                       torch.randn(vec_len, 1) /
                                       torch.sqrt(torch.tensor(vec_len))])
        self.attn_weights.requires_grad = True
        self.attn_weights = nn.Parameter(self.attn_weights)
        self.activation = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)
        self.linear = nn.Linear(vec_len + 1, n_classes)

    def forward(self, input_data):
        hidden = torch.matmul(input_data, self.attn_weights)
        hidden = self.activation(hidden)
        attn = self.softmax(hidden)
        attn = attn.repeat(1, 1, self.vec_len + 1).reshape(attn.shape[0],
                                                           self.seq_len,
                                                           self.vec_len + 1)
        attn_output = input_data * attn
        attn_output = torch.sum(attn_output, axis=1)
        output = self.linear(attn_output)
        return output

Create PyTorch dataset

In [61]:
class TextDataset(torch.utils.data.Dataset):

    def __init__(self, tokens, embeddings, labels):
        """
        :param tokens: List of word tokens
        :param embeddings: Word embeddings (from glove)
        :param labels: List of labels
        """
        self.tokens = tokens
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        emb = torch.tensor(self.embeddings[self.tokens[idx], :])
        input_ = torch.cat((torch.ones(emb.shape[0],1), emb), dim=1)
        return torch.tensor(self.labels[idx]), input_


Function to train the model

In [62]:
def train(train_loader, valid_loader, model, criterion, optimizer,
          device, num_epochs, model_path):
    """
    Function to train the model
    :param train_loader: Data loader for train dataset
    :param valid_loader: Data loader for validation dataset
    :param model: Model object
    :param criterion: Loss function
    :param optimizer: Optimizer
    :param device: CUDA or CPU
    :param num_epochs: Number of epochs
    :param model_path: Path to save the model
    """
    best_loss = 1e8
    for i in range(num_epochs):
        print(f"Epoch {i+1} of {num_epochs}")
        valid_loss, train_loss = [], []
        model.train()
        # Train loop
        for batch_labels, batch_data in tqdm(train_loader):
            # Move data to GPU if available
            batch_labels = batch_labels.to(device)
            batch_data = batch_data.to(device)
            # Forward pass
            batch_output = model(batch_data)
            batch_output = torch.squeeze(batch_output)
            # Calculate loss
            loss = criterion(batch_output, batch_labels)
            train_loss.append(loss.item())
            optimizer.zero_grad()
            # Backward pass
            loss.backward()
            # Gradient update step
            optimizer.step()
        model.eval()
        # Validation loop
        for batch_labels, batch_data in tqdm(valid_loader):
            # Move data to GPU if available
            batch_labels = batch_labels.to(device)
            batch_data = batch_data.to(device)
            # Forward pass
            batch_output = model(batch_data)
            batch_output = torch.squeeze(batch_output)
            # Calculate loss
            loss = criterion(batch_output, batch_labels)
            valid_loss.append(loss.item())
        t_loss = np.mean(train_loss)
        v_loss = np.mean(valid_loss)
        print(f"Train Loss: {t_loss}, Validation Loss: {v_loss}")
        if v_loss < best_loss:
            best_loss = v_loss
            # Save model if validation loss improves
            torch.save(model.state_dict(), model_path)
        print(f"Best Validation Loss: {best_loss}")

Function to test the model

In [63]:
def test(test_loader, model, criterion, device):
    """
    Function to test the model
    :param test_loader: Data loader for test dataset
    :param model: Model object
    :param criterion: Loss function
    :param device: CUDA or CPU
    """
    model.eval()
    test_loss = []
    test_accu = []
    for batch_labels, batch_data in tqdm(test_loader):
        # Move data to device
        batch_labels = batch_labels.to(device)
        batch_data = batch_data.to(device)
        # Forward pass
        batch_output = model(batch_data)
        batch_output = torch.squeeze(batch_output)
        # Calculate loss
        loss = criterion(batch_output, batch_labels)
        test_loss.append(loss.item())
        batch_preds = torch.argmax(batch_output, axis=1)
        # Move predictions to CPU
        if torch.cuda.is_available():
            batch_labels = batch_labels.cpu()
            batch_preds = batch_preds.cpu()
        # Compute accuracy
        test_accu.append(accuracy_score(batch_labels.detach().
                                        numpy(),
                                        batch_preds.detach().
                                        numpy()))
    test_loss = np.mean(test_loss)
    test_accu = np.mean(test_accu)
    print(f"Test Loss: {test_loss}, Test Accuracy: {test_accu}")

###Train attention model

Load the files

In [64]:
tokens = load_file(tokens_path)
labels = load_file(labels_path)
embeddings = load_file(embeddings_path)
label_encoder = load_file(label_encoder_path)
num_classes = len(label_encoder.classes_)
vocabulary = load_file(vocabulary_path)

Split data into train, validation and test sets

In [65]:
X_train, X_test, y_train, y_test = train_test_split(tokens, labels,
                                                    test_size=0.2)
X_train, X_valid, y_train, y_valid = train_test_split(X_train,
                                                      y_train,
                                                      test_size=0.25)

Create PyTorch datasets

In [66]:
train_dataset = TextDataset(X_train, embeddings, y_train)
valid_dataset = TextDataset(X_valid, embeddings, y_valid)
test_dataset = TextDataset(X_test, embeddings, y_test)

Create data loaders

In [67]:
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=16,
                                           shuffle=True,
                                           drop_last=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset,
                                           batch_size=16)
test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=16)


Create model object

In [68]:
device = torch.device("cuda:0" if torch.cuda.is_available()
                      else "cpu")

In [69]:
model = AttentionModel(vec_len, seq_len, num_classes)


Move the model to GPU if available

In [70]:
if torch.cuda.is_available():
    model = model.cuda()

Define loss function and optimizer

In [71]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Training loop

In [72]:
train(train_loader, valid_loader, model, criterion, optimizer,
      device, num_epochs, model_path)

Epoch 1 of 50


100%|██████████| 3473/3473 [00:07<00:00, 465.35it/s]


Train Loss: 0.9218058365563967, Validation Loss: 0.8127043459520469
Best Validation Loss: 0.8127043459520469
Epoch 2 of 50


100%|██████████| 3473/3473 [00:07<00:00, 457.67it/s]


Train Loss: 0.8005242921507824, Validation Loss: 0.7859471900865388
Best Validation Loss: 0.7859471900865388
Epoch 3 of 50


...

Train Loss: 0.7369342622715307, Validation Loss: 0.7461703493740659
Best Validation Loss: 0.7456573980181697
Epoch 49 of 50


100%|██████████| 3473/3473 [00:08<00:00, 419.97it/s]


Train Loss: 0.7368612983581383, Validation Loss: 0.7463930980070435
Best Validation Loss: 0.7456573980181697
Epoch 50 of 50


100%|██████████| 3473/3473 [00:07<00:00, 443.02it/s]

Train Loss: 0.7367762861142955, Validation Loss: 0.7463185629565263
Best Validation Loss: 0.7456573980181697


Test the model

In [73]:
test(test_loader, model, criterion, device)

100%|██████████| 3473/3473 [00:10<00:00, 338.28it/s]

Test Loss: 0.7484082845928619, Test Accuracy: 0.7546985838808471


Predict on new text

In [74]:
input_text = '''I am a victim of Identity Theft & currently have an Experian account that
I can view my Experian Credit Report and getting notified when there is activity on
my Experian Credit Report. For the past 3 days I've spent a total of approximately 9
hours on the phone with Experian. Every time I call I get transferred repeatedly and
then my last transfer and automated message states to press 1 and leave a message and
someone would call me. Every time I press 1 I get an automatic message stating than you
before I even leave a message and get disconnected. I call Experian again, explain what
is happening and the process begins again with the same end result. I was trying to have
this issue attended and resolved informally but I give up after 9 hours. There are hard
hit inquiries on my Experian Credit Report that are fraud, I didn't authorize, or recall
and I respectfully request that Experian remove the hard hit inquiries immediately just
like they've done in the past when I was able to speak to a live Experian representative
in the United States. The following are the hard hit inquiries : BK OF XXXX XX/XX/XXXX
XXXX XXXX XXXX  XX/XX/XXXX XXXX  XXXX XXXX  XX/XX/XXXX XXXX  XX/XX/XXXX XXXX  XXXX
XX/XX/XXXX'''

Process input text

In [75]:
input_text = input_text.lower()
input_text = re.sub(r"[^\w\d'\s]+", " ", input_text)
input_text = re.sub("\d+", "", input_text)
input_text = re.sub(r'[x]{2,}', "", input_text)
input_text = re.sub(' +', ' ', input_text)
tokens = word_tokenize(input_text)

In [76]:
tokens = ['<pad>']*(20-len(tokens))+tokens

In [77]:
idx_token = []
for token in tokens:
    if token in vocabulary:
        idx_token.append(vocabulary.index(token))
    else:
        idx_token.append(vocabulary.index('<unk>'))

In [78]:
token_emb = embeddings[idx_token,:]
token_emb = token_emb[:seq_len, :]
inp = torch.from_numpy(token_emb)

In [79]:
inp = torch.cat((torch.ones(inp.shape[0],1), inp), dim=1)

In [80]:
device = torch.device("cuda:0" if torch.cuda.is_available()
                      else "cpu")

In [81]:
inp = inp.to(device)
inp = torch.unsqueeze(inp, 0)

In [82]:
label_encoder = load_file(label_encoder_path)
num_classes = len(label_encoder.classes_)

In [83]:
import torch

# ذخیره state_dict مدل
torch.save(model.state_dict(), model_path)

In [84]:
# Create model object
model = AttentionModel(vec_len, seq_len, num_classes)

# Load trained weights
model.load_state_dict(torch.load(model_path))

# Move the model to GPU if available
if torch.cuda.is_available():
    model = model.cuda()

# Forward pass
out = torch.squeeze(model(inp))

# Find predicted class
prediction = label_encoder.classes_[torch.argmax(out)]
print(f"Predicted  Class: {prediction}")

Predicted  Class: credit_report


<ipython-input-84-c6eeda1fc3f5>:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))
